# 🏆 BTK Datathon 2026 — v6 OPTIMIZED

**Yarışma Metriği:** MSE  
**Donanım:** NVIDIA CUDA GPU  
**Felsefe:** *Trust CV. Build robust. Validate independently.*

## v5 → v6 Değişiklikleri (Tüm 5 Düzeltme)

| # | Değişiklik | Sebep | Kazanç |
|---|------------|-------|--------|
| 1 | **HGB modeli kaldırıldı** | Ensemble weight=0 (ölçtüm) | ~50 dk |
| 2 | **MLP modeli kaldırıldı** | +0.17 MSE iyileşme için 2.5 saat israfı | ~150 dk |
| 3 | **Phase 4 temizlendi** | 1-row dummy hold artığı vardı | Hız + temizlik |
| 4 | **Akıllı pseudo-labeling** | Holdout'ta iyileşmezse otomatik atlanır | Risk azalır |
| 5 | **Final = 70% Phase4 + 30% Phase1** | Phase4'te daha çok veri var | Daha iyi tahmin |

**Beklenen süre:** v5'in ~3 saatinden v6'da **~30-45 dk** (5x hızlı)  
**Beklenen MSE:** v5 ile aynı seviye (83-85 arası)

## Modeller (5 adet, 4 farklı tür)

| Model | Tür | Optuna Trial |
|-------|-----|--------------|
| XGBoost | Tree | 60 |
| LightGBM | Tree | 60 |
| CatBoost | Tree (native cat) | 40 |
| Ridge+Poly | Linear | 15 |
| KNN top-30 | Distance | 15 |

In [ ]:
# ── Paket kurulumu (Colab için) ──────────────────────────────────────────────
!pip install sentence-transformers -q 2>&1 | tail -1
!pip install xgboost lightgbm catboost optuna -q 2>&1 | tail -1

In [ ]:
import pandas as pd
import numpy as np
import warnings, gc, time
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from scipy.optimize import minimize

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import optuna
from optuna.pruners import SuccessiveHalvingPruner
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED          = 42
N_FOLDS       = 10
N_SEEDS       = 3
HOLDOUT_RATIO = 0.2
TARGET_COL    = 'career_success_score'

np.random.seed(SEED)
print('✓ Kütüphaneler hazır')

In [ ]:
# ── GPU Kontrol ──────────────────────────────────────────────────────────────
GPU_AVAILABLE = False; LGB_GPU = False; CAT_GPU = False; BERT_DEVICE = 'cpu'
tx = np.random.rand(100,5).astype(np.float32); ty = np.random.rand(100).astype(np.float32)

try:
    m = xgb.XGBRegressor(n_estimators=10, device='cuda', tree_method='hist'); m.fit(tx, ty)
    GPU_AVAILABLE = True; print('✓ XGBoost GPU')
except: print('✗ XGBoost CPU')

try:
    m = lgb.LGBMRegressor(n_estimators=10, device='gpu', verbose=-1); m.fit(tx, ty)
    LGB_GPU = True; print('✓ LightGBM GPU')
except: print('✗ LightGBM CPU')

try:
    m = CatBoostRegressor(iterations=10, task_type='GPU', devices='0', verbose=0); m.fit(tx, ty)
    CAT_GPU = True; print('✓ CatBoost GPU')
except: print('✗ CatBoost CPU')

try:
    import torch
    if torch.cuda.is_available():
        BERT_DEVICE = 'cuda'; print(f'✓ PyTorch CUDA: {torch.cuda.get_device_name(0)}')
    else: print('✗ PyTorch CPU')
except: print('✗ PyTorch yok')

xgb_gpu = lambda: {'device':'cuda','tree_method':'hist'} if GPU_AVAILABLE else {'tree_method':'hist','n_jobs':-1}
lgb_gpu = lambda: {'device':'gpu'} if LGB_GPU else {'n_jobs':-1}
cat_gpu = lambda: {'task_type':'GPU','devices':'0'} if CAT_GPU else {}

## 1. Veri + 🔒 80/20 Hold-out Split

In [ ]:
TRAIN_PATH  = 'train.csv'
TEST_PATH   = 'test_x.csv'
SAMPLE_PATH = 'sample_submission.csv'

train_full = pd.read_csv(TRAIN_PATH)
test       = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_PATH)

y_full = train_full[TARGET_COL].values
y_bin_full = pd.qcut(y_full, q=10, labels=False, duplicates='drop')

dev_idx, holdout_idx = train_test_split(
    np.arange(len(train_full)), test_size=HOLDOUT_RATIO,
    random_state=SEED, stratify=y_bin_full)

train_dev  = train_full.iloc[dev_idx].reset_index(drop=True)
train_hold = train_full.iloc[holdout_idx].reset_index(drop=True)
y_dev      = y_full[dev_idx]
y_hold     = y_full[holdout_idx]

print(f'Full train: {train_full.shape}')
print(f'Dev: {train_dev.shape}  | Holdout: {train_hold.shape}  | Test: {test.shape}')
print(f'Dev y μ={y_dev.mean():.2f} σ={y_dev.std():.2f}')
print(f'Hold y μ={y_hold.mean():.2f} σ={y_hold.std():.2f}  ← yakın olmalı')

y_dev_bin = pd.qcut(y_dev, q=10, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_SPLITS = list(skf.split(train_dev, y_dev_bin))
test_ids = test['student_id'].values

## 2. 🎯 Role-Skill Engineering

In [ ]:
ROLE_PROFILES = {
    'Cloud Engineer':        {'cloud_score':3, 'devops_score':2, 'problem_solving_score':1},
    'DevOps Engineer':       {'devops_score':3, 'cloud_score':2, 'problem_solving_score':1},
    'MLOps Engineer':        {'devops_score':2, 'machine_learning_score':2, 'cloud_score':1, 'problem_solving_score':1},
    'Frontend Developer':    {'frontend_score':3, 'coding_score':2, 'problem_solving_score':1},
    'Backend Developer':     {'backend_score':3, 'coding_score':2, 'sql_score':1, 'problem_solving_score':1},
    'Software Developer':    {'coding_score':2, 'problem_solving_score':2, 'data_structures_score':2, 'backend_score':1, 'frontend_score':1},
    'Data Scientist':        {'machine_learning_score':3, 'sql_score':2, 'problem_solving_score':2, 'data_structures_score':1},
    'AI Engineer':           {'machine_learning_score':3, 'coding_score':2, 'problem_solving_score':1, 'data_structures_score':1},
    'Data Analyst':          {'sql_score':3, 'data_structures_score':1, 'problem_solving_score':1},
    'Product Analyst':       {'sql_score':2, 'problem_solving_score':2},
    'Cybersecurity Analyst': {'devops_score':2, 'problem_solving_score':2, 'coding_score':1},
}
PRIMARY_SKILL = {r: max(w.items(), key=lambda x: x[1])[0] for r, w in ROLE_PROFILES.items()}

def add_role_features(df):
    df = df.copy()
    df['matched_skill'] = 0.0
    for role, skill in PRIMARY_SKILL.items():
        m = df['target_role'] == role
        df.loc[m, 'matched_skill'] = df.loc[m, skill].values
    df['role_composite'] = 0.0
    for role, w in ROLE_PROFILES.items():
        m = df['target_role'] == role
        if m.sum() == 0: continue
        tot = sum(w.values())
        df.loc[m, 'role_composite'] = (sum(df.loc[m, sk]*wt for sk, wt in w.items()) / tot).values
    return df

train_dev  = add_role_features(train_dev)
train_hold = add_role_features(train_hold)
train_full = add_role_features(train_full)  # Phase 4 için
test       = add_role_features(test)
print('✓ Role-skill features eklendi')

## 3. 🧠 NLP — BERT + TF-IDF

In [ ]:
BERT_OK = False
# Full corpus: full_train + test (Phase 4 retrain için ihtiyaç var)
all_texts_full = pd.concat([
    train_full['mentor_feedback_text'],
    test['mentor_feedback_text']
]).fillna('').reset_index(drop=True)
n_train, n_test = len(train_full), len(test)

try:
    from sentence_transformers import SentenceTransformer
    print('Sentence-BERT yükleniyor...')
    bert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=BERT_DEVICE)
    print(f'Encoding {len(all_texts_full)} text on {BERT_DEVICE}...')
    t0 = time.time()
    emb = bert_model.encode(all_texts_full.tolist(), batch_size=128, show_progress_bar=True,
                              convert_to_numpy=True, device=BERT_DEVICE)
    print(f'BERT: {emb.shape} in {time.time()-t0:.1f}s')
    pca = PCA(n_components=64, random_state=SEED)
    bert_reduced = pca.fit_transform(emb)
    print(f'PCA → 64-dim, explained: {pca.explained_variance_ratio_.sum():.3f}')
    
    bert_cols = [f'bert_{i}' for i in range(64)]
    # train_full sırasına göre (dev_idx ve holdout_idx birleşik)
    bert_full_df = pd.DataFrame(bert_reduced[:n_train], columns=bert_cols)
    bert_test    = pd.DataFrame(bert_reduced[n_train:], columns=bert_cols)
    
    # Dev ve hold için ayrıştır
    bert_dev  = bert_full_df.iloc[dev_idx].reset_index(drop=True)
    bert_hold = bert_full_df.iloc[holdout_idx].reset_index(drop=True)
    
    del bert_model, emb; gc.collect()
    if BERT_DEVICE == 'cuda':
        import torch; torch.cuda.empty_cache()
    BERT_OK = True
    print('✓ BERT embeddings hazır')
except Exception as e:
    print(f'⚠ BERT skip: {str(e)[:100]}')
    bert_dev = bert_hold = bert_test = bert_full_df = pd.DataFrame()

In [ ]:
# TF-IDF word + char
word_tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=3, sublinear_tf=True, analyzer='word')
word_m = word_tfidf.fit_transform(all_texts_full)
word_feats = TruncatedSVD(n_components=50, random_state=SEED).fit_transform(word_m)

char_tfidf = TfidfVectorizer(max_features=3000, ngram_range=(3,5), min_df=3, sublinear_tf=True, analyzer='char_wb')
char_m = char_tfidf.fit_transform(all_texts_full)
char_feats = TruncatedSVD(n_components=30, random_state=SEED).fit_transform(char_m)

nlp_cols = [f'nlp_w_{i}' for i in range(50)] + [f'nlp_c_{i}' for i in range(30)]
nlp_all = pd.DataFrame(np.hstack([word_feats, char_feats]), columns=nlp_cols)
nlp_all['text_len']   = all_texts_full.str.len().values
nlp_all['word_count'] = all_texts_full.str.split().str.len().values

nlp_full_df = nlp_all.iloc[:n_train].reset_index(drop=True)
nlp_test    = nlp_all.iloc[n_train:].reset_index(drop=True)
nlp_dev     = nlp_full_df.iloc[dev_idx].reset_index(drop=True)
nlp_hold    = nlp_full_df.iloc[holdout_idx].reset_index(drop=True)
print(f'✓ TF-IDF features: {nlp_dev.shape[1]}')

## 4. Target Encoding (Sızıntısız)

In [ ]:
def kfold_te_dev(df_dev, df_hold, df_test, col, y_dev, splits, smoothing=20):
    """Phase 1: dev y kullanılarak hold + test encode."""
    gm = y_dev.mean()
    dev_t = df_dev.copy(); dev_t['__y__'] = y_dev
    oof = np.zeros(len(df_dev))
    for tr_idx, val_idx in splits:
        tr = dev_t.iloc[tr_idx]
        agg = tr.groupby(col)['__y__'].agg(['mean','count'])
        agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
        val = dev_t.iloc[val_idx]
        oof[val_idx] = val[col].map(agg['s']).fillna(gm).values
    agg = dev_t.groupby(col)['__y__'].agg(['mean','count'])
    agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
    hold_e = df_hold[col].map(agg['s']).fillna(gm).values
    test_e = df_test[col].map(agg['s']).fillna(gm).values
    return oof, hold_e, test_e

def kfold_te_full(df_full, df_test, col, y_full, splits, smoothing=20):
    """Phase 4: full train y kullanılarak test encode."""
    gm = y_full.mean()
    full_t = df_full.copy(); full_t['__y__'] = y_full
    oof = np.zeros(len(df_full))
    for tr_idx, val_idx in splits:
        tr = full_t.iloc[tr_idx]
        agg = tr.groupby(col)['__y__'].agg(['mean','count'])
        agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
        val = full_t.iloc[val_idx]
        oof[val_idx] = val[col].map(agg['s']).fillna(gm).values
    agg = full_t.groupby(col)['__y__'].agg(['mean','count'])
    agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
    test_e = df_test[col].map(agg['s']).fillna(gm).values
    return oof, test_e

# Phase 1: dev TE
te_dev, te_hold, te_test_p1 = {}, {}, {}
for col in ['department','target_role','hobby','preferred_social_media_platform','university_tier']:
    oo, ho, te = kfold_te_dev(train_dev, train_hold, test, col, y_dev, FOLD_SPLITS, 20)
    te_dev[f'te_{col}'] = oo; te_hold[f'te_{col}'] = ho; te_test_p1[f'te_{col}'] = te

# Çapraz TE
for c1, c2 in [('department','target_role'), ('university_tier','target_role'), ('target_role','hobby')]:
    combo = f'{c1}_x_{c2}'
    train_dev[combo]  = train_dev[c1].astype(str)+'|'+train_dev[c2].astype(str)
    train_hold[combo] = train_hold[c1].astype(str)+'|'+train_hold[c2].astype(str)
    test[combo]       = test[c1].astype(str)+'|'+test[c2].astype(str)
    oo, ho, te = kfold_te_dev(train_dev, train_hold, test, combo, y_dev, FOLD_SPLITS, 30)
    te_dev[f'te_{combo}'] = oo; te_hold[f'te_{combo}'] = ho; te_test_p1[f'te_{combo}'] = te
    train_dev.drop(columns=[combo], inplace=True)
    train_hold.drop(columns=[combo], inplace=True)
    test.drop(columns=[combo], inplace=True)

train_te_p1 = pd.DataFrame(te_dev)
hold_te_p1  = pd.DataFrame(te_hold)
test_te_p1  = pd.DataFrame(te_test_p1)
print(f'✓ Phase 1 TE features: {train_te_p1.shape[1]}')

## 5. Feature Engineering

In [ ]:
TIER_MAP  = {'Tier 1':4,'Tier 2':3,'Tier 3':2,'Tier 4':1}
TECH_COLS = ['coding_score','problem_solving_score','data_structures_score',
             'sql_score','machine_learning_score','backend_score','frontend_score',
             'cloud_score','devops_score']
SOFT_COLS = ['communication_score','teamwork_score','leadership_score','presentation_score']
CAT_COLS  = ['department','target_role','hobby','preferred_social_media_platform']

def derive_features(df):
    df = df.copy()
    df['tech_mean']  = df[TECH_COLS].mean(axis=1)
    df['tech_max']   = df[TECH_COLS].max(axis=1)
    df['tech_min']   = df[TECH_COLS].min(axis=1)
    df['tech_std']   = df[TECH_COLS].std(axis=1)
    df['tech_sum']   = df[TECH_COLS].sum(axis=1)
    df['tech_top3']  = df[TECH_COLS].apply(lambda r: r.nlargest(3).mean(), axis=1)
    df['tech_bot3']  = df[TECH_COLS].apply(lambda r: r.nsmallest(3).mean(), axis=1)
    df['soft_mean'] = df[SOFT_COLS].mean(axis=1)
    df['soft_sum']  = df[SOFT_COLS].sum(axis=1)
    df['soft_std']  = df[SOFT_COLS].std(axis=1)
    # Role interactions
    df['matched_x_proj']        = df['matched_skill'] * df['project_quality_score']
    df['matched_x_interview']   = df['matched_skill'] * df['technical_interview_score']
    df['role_comp_x_proj']      = df['role_composite'] * df['project_quality_score']
    df['role_comp_x_interview'] = df['role_composite'] * df['technical_interview_score']
    df['matched_minus_tech']    = df['matched_skill'] - df['tech_mean']
    df['role_comp_minus_avg']   = df['role_composite'] - df['tech_mean']
    # Mülakat
    df['interview_composite'] = df['technical_interview_score']*0.6 + df['hr_interview_score']*0.4
    df['interview_diff']      = df['technical_interview_score'] - df['hr_interview_score']
    # Deneyim
    df['total_projects']  = df['real_client_project_count'] + df['freelance_project_count']
    df['experience_score']= (df['internship_count']*3 + df['real_client_project_count']*2 + df['freelance_project_count'])
    df['exp_per_year']    = df['experience_score'] / (df['age'] - 18).clip(lower=1)
    # GitHub
    df['github_score']    = (df['github_repo_count'] * np.log1p(df['github_avg_stars']+1) + df['open_source_contribution_count'])
    df['oss_ratio']       = df['open_source_contribution_count'] / (df['github_repo_count']+1)
    # Profil
    df['profile_composite']= (df['portfolio_score']+df['linkedin_profile_score']+df['cv_quality_score'])/3
    df['profile_min']      = df[['portfolio_score','linkedin_profile_score','cv_quality_score']].min(axis=1)
    df['profile_max']      = df[['portfolio_score','linkedin_profile_score','cv_quality_score']].max(axis=1)
    # Hackathon
    df['hackathon_eff']   = np.where(df['hackathon_count']>0, df['hackathon_awards']/df['hackathon_count'], 0)
    df['app_success_rate']= df['interviews_attended'] / (df['applications_sent']+1)
    # Üni
    df['university_tier_num'] = df['university_tier'].map(TIER_MAP)
    df['years_to_grad'] = df['graduation_year'] - df['application_year']
    df['cert_per_internship'] = df['certification_count'] / (df['internship_count']+1)
    df['cgpa_x_attendance']   = df['cgpa'] * df['attendance_rate']
    df['failed_penalty']      = df['failed_courses_count'] / (df['cgpa']+0.01)
    # Genel interactions
    df['soft_x_technical']    = df['soft_mean'] * df['technical_interview_score']
    df['proj_x_interview']    = df['project_quality_score'] * df['technical_interview_score']
    df['proj_x_portfolio']    = df['project_quality_score'] * df['portfolio_score']
    # Composite
    df['overall_score'] = (df['tech_mean']*0.25 + df['soft_mean']*0.15 +
                            df['project_quality_score']*0.25 + df['profile_composite']*0.10 +
                            df['interview_composite']*0.15 + df['matched_skill']*0.10)
    return df

def prepare_features(df_train_like, df_test, fill_medians=None, label_encoders=None):
    """Generic feature preparation: nulls → derive → label encoding."""
    df_train_like = df_train_like.copy(); df_test = df_test.copy()
    fill_cols = ['english_exam_score','internship_duration_months','portfolio_score',
                 'github_avg_stars','open_source_contribution_count',
                 'linkedin_profile_score','hr_interview_score']
    if fill_medians is None:
        fill_medians = {c: df_train_like[c].median() for c in fill_cols}
    for c in fill_cols:
        df_train_like[c] = df_train_like[c].fillna(fill_medians[c])
        df_test[c]       = df_test[c].fillna(fill_medians[c])
    
    df_train_like = derive_features(df_train_like)
    df_test       = derive_features(df_test)
    df_train_cat  = df_train_like.copy()
    df_test_cat   = df_test.copy()
    
    if label_encoders is None:
        label_encoders = {}
        for col in CAT_COLS + ['university_tier']:
            le = LabelEncoder()
            le.fit(pd.concat([df_train_like[col], df_test[col]]).astype(str))
            label_encoders[col] = le
    for col, le in label_encoders.items():
        df_train_like[col] = le.transform(df_train_like[col].astype(str))
        df_test[col]       = le.transform(df_test[col].astype(str))
    
    drop_cols = ['student_id','mentor_feedback_text']
    if TARGET_COL in df_train_like.columns: drop_cols.append(TARGET_COL)
    df_train_like = df_train_like.drop(columns=drop_cols, errors='ignore')
    df_test       = df_test.drop(columns=drop_cols, errors='ignore')
    df_train_cat  = df_train_cat.drop(columns=drop_cols, errors='ignore')
    df_test_cat   = df_test_cat.drop(columns=drop_cols, errors='ignore')
    
    return df_train_like, df_test, df_train_cat, df_test_cat, fill_medians, label_encoders

# Phase 1: dev üzerinde fit
X_dev_base, X_test_base_p1, X_dev_cat, X_test_cat_p1, dev_medians, dev_les = prepare_features(train_dev, test)
X_hold_base, _, X_hold_cat, _, _, _ = prepare_features(train_hold, test, fill_medians=dev_medians, label_encoders=dev_les)
print(f'Phase 1 base shape: dev {X_dev_base.shape}, hold {X_hold_base.shape}, test {X_test_base_p1.shape}')

In [ ]:
# Tüm parçaları birleştir
def merge_all(base, nlp, te, bert):
    parts = [base.reset_index(drop=True), nlp.reset_index(drop=True), te.reset_index(drop=True)]
    if BERT_OK and len(bert) > 0:
        parts.append(bert.reset_index(drop=True))
    return pd.concat(parts, axis=1)

X_dev      = merge_all(X_dev_base, nlp_dev, train_te_p1, bert_dev)
X_hold     = merge_all(X_hold_base, nlp_hold, hold_te_p1, bert_hold)
X_test_p1  = merge_all(X_test_base_p1, nlp_test, test_te_p1, bert_test)
X_dev_cat  = merge_all(X_dev_cat, nlp_dev, train_te_p1, bert_dev)
X_hold_cat = merge_all(X_hold_cat, nlp_hold, hold_te_p1, bert_hold)
X_test_cat_p1 = merge_all(X_test_cat_p1, nlp_test, test_te_p1, bert_test)

# Null doldurma (dev medianı)
col_medians = X_dev.median(numeric_only=True)
X_dev      = X_dev.fillna(col_medians)
X_hold     = X_hold.fillna(col_medians)
X_test_p1  = X_test_p1.fillna(col_medians)
for c in X_dev_cat.columns:
    if pd.api.types.is_numeric_dtype(X_dev_cat[c]) and X_dev_cat[c].isna().any():
        X_dev_cat[c]  = X_dev_cat[c].fillna(col_medians.get(c, 0))
        X_hold_cat[c] = X_hold_cat[c].fillna(col_medians.get(c, 0))
        X_test_cat_p1[c] = X_test_cat_p1[c].fillna(col_medians.get(c, 0))

cat_features_idx = [X_dev_cat.columns.get_loc(c) for c in CAT_COLS + ['university_tier']]
print(f'✓ Final Phase 1 features: Dev {X_dev.shape}, Hold {X_hold.shape}, Test {X_test_p1.shape}')

## 6. 🔍 Feature Selection (XGB Importance)

In [ ]:
print('Quick XGB ile feature importance...')
quick = xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, random_state=SEED, **xgb_gpu())
quick.fit(X_dev, y_dev)
imps = pd.Series(quick.feature_importances_, index=X_dev.columns).sort_values(ascending=False)

all_features_keep = imps[imps > imps.quantile(0.15)].index.tolist()
top_30_features = imps.head(30).index.tolist()
top_10_features = imps.head(10).index.tolist()
print(f'Trees için: {X_dev.shape[1]} → {len(all_features_keep)} feature')
print(f'KNN için   : top 30')
print(f'Ridge için : top 10  →  {top_10_features}')

# Standardize
scaler = StandardScaler()
X_dev_std  = pd.DataFrame(scaler.fit_transform(X_dev), columns=X_dev.columns, index=X_dev.index)
X_hold_std = pd.DataFrame(scaler.transform(X_hold), columns=X_hold.columns, index=X_hold.index)
X_test_std_p1 = pd.DataFrame(scaler.transform(X_test_p1), columns=X_test_p1.columns, index=X_test_p1.index)

X_dev_sel = X_dev[all_features_keep]
X_hold_sel = X_hold[all_features_keep]
X_test_sel_p1 = X_test_p1[all_features_keep]
X_dev_cat_sel = X_dev_cat[all_features_keep]
X_hold_cat_sel = X_hold_cat[all_features_keep]
X_test_cat_sel_p1 = X_test_cat_p1[all_features_keep]
cat_idx_sel = [X_dev_cat_sel.columns.get_loc(c) for c in CAT_COLS + ['university_tier'] if c in X_dev_cat_sel.columns]

X_dev_knn = X_dev_std[top_30_features]
X_hold_knn = X_hold_std[top_30_features]
X_test_knn_p1 = X_test_std_p1[top_30_features]

X_dev_poly = X_dev[top_10_features]
X_hold_poly = X_hold[top_10_features]
X_test_poly_p1 = X_test_p1[top_10_features]

## 7. Optuna (XGB, LGB, CAT, Ridge, KNN)

In [ ]:
opt_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
OPT_SPLITS = list(opt_skf.split(X_dev, y_dev_bin))

In [ ]:
# XGBoost
def xgb_obj(trial):
    p = {'n_estimators': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma': trial.suggest_float('gamma', 0, 2.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'early_stopping_rounds': 50, 'random_state': SEED, **xgb_gpu()}
    rmses = []
    for i, (tr, va) in enumerate(OPT_SPLITS):
        m = xgb.XGBRegressor(**p)
        m.fit(X_dev_sel.iloc[tr], y_dev[tr], eval_set=[(X_dev_sel.iloc[va], y_dev[va])], verbose=False)
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_sel.iloc[va]))))
        trial.report(np.mean(rmses), i)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(rmses)

print('XGBoost (60 trial)...')
t0 = time.time()
xgb_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED), pruner=SuccessiveHalvingPruner())
xgb_study.optimize(xgb_obj, n_trials=60, show_progress_bar=True)
print(f'XGB: {xgb_study.best_value:.4f}  ({time.time()-t0:.0f}s)')

In [ ]:
# LightGBM
def lgb_obj(trial):
    p = {'n_estimators': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 400),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'random_state': SEED, 'verbose': -1, **lgb_gpu()}
    rmses = []
    for i, (tr, va) in enumerate(OPT_SPLITS):
        m = lgb.LGBMRegressor(**p)
        m.fit(X_dev_sel.iloc[tr], y_dev[tr], eval_set=[(X_dev_sel.iloc[va], y_dev[va])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_sel.iloc[va]))))
        trial.report(np.mean(rmses), i)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(rmses)

print('LightGBM (60 trial)...')
t0 = time.time()
lgb_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED), pruner=SuccessiveHalvingPruner())
lgb_study.optimize(lgb_obj, n_trials=60, show_progress_bar=True)
print(f'LGB: {lgb_study.best_value:.4f}  ({time.time()-t0:.0f}s)')

In [ ]:
# CatBoost
def cat_obj(trial):
    p = {'iterations': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 100),
        'random_strength': trial.suggest_float('random_strength', 0.5, 5),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_seed': SEED, 'verbose': 0, 'loss_function': 'RMSE',
        'early_stopping_rounds': 50, **cat_gpu()}
    rmses = []
    for i, (tr, va) in enumerate(OPT_SPLITS):
        m = CatBoostRegressor(**p)
        m.fit(X_dev_cat_sel.iloc[tr], y_dev[tr], eval_set=(X_dev_cat_sel.iloc[va], y_dev[va]),
              cat_features=cat_idx_sel, verbose=0)
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_cat_sel.iloc[va]))))
        trial.report(np.mean(rmses), i)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(rmses)

print('CatBoost (40 trial)...')
t0 = time.time()
cat_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED), pruner=SuccessiveHalvingPruner())
cat_study.optimize(cat_obj, n_trials=40, show_progress_bar=True)
print(f'CAT: {cat_study.best_value:.4f}  ({time.time()-t0:.0f}s)')

In [ ]:
# Ridge + Polynomial
def ridge_obj(trial):
    p = {'alpha': trial.suggest_float('alpha', 0.1, 100, log=True),
         'degree': trial.suggest_int('degree', 2, 3)}
    rmses = []
    for tr, va in OPT_SPLITS:
        pipe = Pipeline([('s', StandardScaler()),
                          ('p', PolynomialFeatures(degree=p['degree'], include_bias=False)),
                          ('r', Ridge(alpha=p['alpha'], random_state=SEED))])
        pipe.fit(X_dev_poly.iloc[tr], y_dev[tr])
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], pipe.predict(X_dev_poly.iloc[va]))))
    return np.mean(rmses)

print('Ridge+Poly (15 trial)...')
ridge_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
ridge_study.optimize(ridge_obj, n_trials=15, show_progress_bar=True)
print(f'Ridge: {ridge_study.best_value:.4f}')

In [ ]:
# KNN top-30
def knn_obj(trial):
    p = {'n_neighbors': trial.suggest_int('n_neighbors', 10, 100),
         'weights': trial.suggest_categorical('weights', ['uniform','distance']),
         'p': trial.suggest_int('p', 1, 2)}
    rmses = []
    for tr, va in OPT_SPLITS:
        m = KNeighborsRegressor(**p, n_jobs=-1)
        m.fit(X_dev_knn.iloc[tr], y_dev[tr])
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_knn.iloc[va]))))
    return np.mean(rmses)

print('KNN (15 trial)...')
knn_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
knn_study.optimize(knn_obj, n_trials=15, show_progress_bar=True)
print(f'KNN: {knn_study.best_value:.4f}')

## 8. Phase 1 Stacking (5 model × 10-fold × 3-seed)

In [ ]:
def stack_p1(factory, splits, X_tr, X_ho, X_te, y_tr, n_seeds=N_SEEDS, kind='plain', cat_idx=None):
    oof = np.zeros(len(y_tr)); ho_p = np.zeros(len(X_ho)); te_p = np.zeros(len(X_te))
    per_fold_mse = []
    for seed in range(n_seeds):
        fold_ho = np.zeros(len(X_ho)); fold_te = np.zeros(len(X_te))
        for fold_i, (tr_idx, val_idx) in enumerate(splits):
            m = factory(seed)
            if kind == 'xgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])], verbose=False)
            elif kind == 'lgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            elif kind == 'cat':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=(X_tr.iloc[val_idx], y_tr[val_idx]),
                      cat_features=cat_idx, verbose=0)
            else:
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx])
            val_pred = m.predict(X_tr.iloc[val_idx])
            oof[val_idx] += val_pred / n_seeds
            fold_ho += m.predict(X_ho) / len(splits)
            fold_te += m.predict(X_te) / len(splits)
            if seed == 0:
                per_fold_mse.append(mean_squared_error(y_tr[val_idx], val_pred))
        ho_p += fold_ho / n_seeds
        te_p += fold_te / n_seeds
    return oof, ho_p, te_p, per_fold_mse

print(f'Phase 1 Stacking ({N_FOLDS}-fold × {N_SEEDS}-seed × 5 model)...\n')
stability = {}

# Model factories
xgb_factory = lambda s: xgb.XGBRegressor(**xgb_study.best_params, random_state=SEED+s, **xgb_gpu(),
                                            early_stopping_rounds=50, n_estimators=3000)
lgb_factory = lambda s: lgb.LGBMRegressor(**lgb_study.best_params, random_state=SEED+s, **lgb_gpu(),
                                            n_estimators=3000, verbose=-1)
cat_factory = lambda s: CatBoostRegressor(**cat_study.best_params, random_seed=SEED+s, **cat_gpu(),
                                            iterations=3000, verbose=0, loss_function='RMSE',
                                            early_stopping_rounds=50)
best_ridge = ridge_study.best_params
ridge_factory = lambda s: Pipeline([('s', StandardScaler()),
    ('p', PolynomialFeatures(degree=best_ridge['degree'], include_bias=False)),
    ('r', Ridge(alpha=best_ridge['alpha'], random_state=SEED+s))])
knn_factory = lambda s: KNeighborsRegressor(**knn_study.best_params, n_jobs=-1)

t0 = time.time()
xgb_oof, xgb_ho, xgb_te, xgb_fmse = stack_p1(xgb_factory, FOLD_SPLITS, X_dev_sel, X_hold_sel, X_test_sel_p1, y_dev, kind='xgb')
stability['XGB'] = xgb_fmse
print(f'  XGB   : OOF MSE={mean_squared_error(y_dev, xgb_oof):.3f}  Hold MSE={mean_squared_error(y_hold, xgb_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
lgb_oof, lgb_ho, lgb_te, lgb_fmse = stack_p1(lgb_factory, FOLD_SPLITS, X_dev_sel, X_hold_sel, X_test_sel_p1, y_dev, kind='lgb')
stability['LGB'] = lgb_fmse
print(f'  LGB   : OOF MSE={mean_squared_error(y_dev, lgb_oof):.3f}  Hold MSE={mean_squared_error(y_hold, lgb_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
cat_oof, cat_ho, cat_te, cat_fmse = stack_p1(cat_factory, FOLD_SPLITS, X_dev_cat_sel, X_hold_cat_sel, X_test_cat_sel_p1, y_dev, kind='cat', cat_idx=cat_idx_sel)
stability['CAT'] = cat_fmse
print(f'  CAT   : OOF MSE={mean_squared_error(y_dev, cat_oof):.3f}  Hold MSE={mean_squared_error(y_hold, cat_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
ridge_oof, ridge_ho, ridge_te, ridge_fmse = stack_p1(ridge_factory, FOLD_SPLITS, X_dev_poly, X_hold_poly, X_test_poly_p1, y_dev, n_seeds=1)
stability['Ridge'] = ridge_fmse
print(f'  Ridge : OOF MSE={mean_squared_error(y_dev, ridge_oof):.3f}  Hold MSE={mean_squared_error(y_hold, ridge_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
knn_oof, knn_ho, knn_te, knn_fmse = stack_p1(knn_factory, FOLD_SPLITS, X_dev_knn, X_hold_knn, X_test_knn_p1, y_dev, n_seeds=1)
stability['KNN'] = knn_fmse
print(f'  KNN   : OOF MSE={mean_squared_error(y_dev, knn_oof):.3f}  Hold MSE={mean_squared_error(y_hold, knn_ho):.3f}  ({time.time()-t0:.0f}s)')

oof_matrix     = np.column_stack([xgb_oof, lgb_oof, cat_oof, ridge_oof, knn_oof])
holdout_matrix = np.column_stack([xgb_ho, lgb_ho, cat_ho, ridge_ho, knn_ho])
test_matrix_p1 = np.column_stack([xgb_te, lgb_te, cat_te, ridge_te, knn_te])
model_names    = ['XGB','LGB','CAT','Ridge','KNN']
n_models       = 5

## 9. 🔬 Stability Check

In [ ]:
print('Per-fold MSE varyans analizi:\n')
for name, fmse in stability.items():
    mean_mse = np.mean(fmse); std_mse = np.std(fmse); cv = std_mse/mean_mse*100
    flag = '✓ STABLE' if cv < 5 else '⚠️ UNSTABLE' if cv < 10 else '🚨 VERY UNSTABLE'
    print(f'  {name:<8} | mean={mean_mse:6.3f} | std={std_mse:5.3f} | CV={cv:5.2f}%  {flag}')

## 10. Weight Optimization + Holdout Doğrulama

In [ ]:
def weighted_rmse(w, preds, y_true):
    return np.sqrt(mean_squared_error(y_true, preds @ w))

constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = [(0.0, 1.0)] * n_models

# SLSQP weighted
best_loss, best_w = float('inf'), None
for trial in range(100):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_matrix, y_dev),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_loss: best_loss, best_w = res.fun, res.x

# Ridge meta
best_ridge_alpha, best_ridge_meta_loss = None, float('inf')
meta_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for alpha in [0.01, 0.1, 1, 5, 10, 50, 100]:
    fold_mses = []
    for tr_idx, val_idx in meta_kf.split(oof_matrix, y_dev_bin):
        meta = Ridge(alpha=alpha); meta.fit(oof_matrix[tr_idx], y_dev[tr_idx])
        fold_mses.append(np.sqrt(mean_squared_error(y_dev[val_idx], meta.predict(oof_matrix[val_idx]))))
    avg = np.mean(fold_mses)
    if avg < best_ridge_meta_loss: best_ridge_meta_loss, best_ridge_alpha = avg, alpha

ridge_meta = Ridge(alpha=best_ridge_alpha); ridge_meta.fit(oof_matrix, y_dev)

# Holdout karşılaştırma
slsqp_ho_mse = mean_squared_error(y_hold, holdout_matrix @ best_w)
ridge_ho_mse = mean_squared_error(y_hold, ridge_meta.predict(holdout_matrix))
blend_ho_mse = mean_squared_error(y_hold, 0.5*(holdout_matrix @ best_w) + 0.5*ridge_meta.predict(holdout_matrix))

print('═'*60)
print('  ENSEMBLE YÖNTEMLERİ — Holdout MSE')
print('═'*60)
print(f'  SLSQP weighted     : {slsqp_ho_mse:.3f}')
print(f'  Ridge meta (α={best_ridge_alpha}) : {ridge_ho_mse:.3f}')
print(f'  50/50 Blend        : {blend_ho_mse:.3f}')
print('═'*60)

for n, w in zip(model_names, best_w):
    print(f'  weight {n:<8}: {w:.4f}')

options = [
    ('SLSQP', slsqp_ho_mse, lambda M: M @ best_w),
    ('Ridge', ridge_ho_mse, lambda M: ridge_meta.predict(M)),
    ('Blend', blend_ho_mse, lambda M: 0.5*(M @ best_w) + 0.5*ridge_meta.predict(M))
]
options.sort(key=lambda x: x[1])
best_method_name, best_method_holdout, best_method_fn = options[0]
print(f'\n🏆 En iyi: {best_method_name}  (Holdout MSE = {best_method_holdout:.3f})')

## 11. 🔄 Akıllı Pseudo-Labeling (Sadece holdout iyileşirse kullanılır)

In [ ]:
test_stds = test_matrix_p1.std(axis=1)
PSEUDO_FRAC = 0.10  # Konservatif: sadece %10
n_pseudo = int(len(test) * PSEUDO_FRAC)
conf_idx = np.argsort(test_stds)[:n_pseudo]
pseudo_labels = np.clip(best_method_fn(test_matrix_p1)[conf_idx], 0, 100)

print(f'Pseudo-label: {n_pseudo} test örneği (en güvenilir %{PSEUDO_FRAC*100:.0f})')

# Augmented dev (pseudo-eklenmiş)
X_dev_aug      = pd.concat([X_dev_sel, X_test_sel_p1.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_dev_cat_aug  = pd.concat([X_dev_cat_sel, X_test_cat_sel_p1.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_dev_poly_aug = pd.concat([X_dev_poly, X_test_poly_p1.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_dev_knn_aug  = pd.concat([X_dev_knn, X_test_knn_p1.iloc[conf_idx]], axis=0).reset_index(drop=True)
y_dev_aug      = np.concatenate([y_dev, pseudo_labels])
y_dev_aug_bin  = pd.qcut(y_dev_aug, q=10, labels=False, duplicates='drop')
FOLD_SPLITS_AUG = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_dev_aug, y_dev_aug_bin))

n_orig = len(y_dev)
t0 = time.time()
xgb_oof2, xgb_ho2, xgb_te2, _ = stack_p1(xgb_factory, FOLD_SPLITS_AUG, X_dev_aug, X_hold_sel, X_test_sel_p1, y_dev_aug, kind='xgb')
lgb_oof2, lgb_ho2, lgb_te2, _ = stack_p1(lgb_factory, FOLD_SPLITS_AUG, X_dev_aug, X_hold_sel, X_test_sel_p1, y_dev_aug, kind='lgb')
cat_oof2, cat_ho2, cat_te2, _ = stack_p1(cat_factory, FOLD_SPLITS_AUG, X_dev_cat_aug, X_hold_cat_sel, X_test_cat_sel_p1, y_dev_aug, kind='cat', cat_idx=cat_idx_sel)
ridge_oof2, ridge_ho2, ridge_te2, _ = stack_p1(ridge_factory, FOLD_SPLITS_AUG, X_dev_poly_aug, X_hold_poly, X_test_poly_p1, y_dev_aug, n_seeds=1)
knn_oof2, knn_ho2, knn_te2, _ = stack_p1(knn_factory, FOLD_SPLITS_AUG, X_dev_knn_aug, X_hold_knn, X_test_knn_p1, y_dev_aug, n_seeds=1)
print(f'Pseudo stacking: {time.time()-t0:.0f}s')

oof_matrix_aug    = np.column_stack([xgb_oof2[:n_orig], lgb_oof2[:n_orig], cat_oof2[:n_orig], ridge_oof2[:n_orig], knn_oof2[:n_orig]])
holdout_matrix_aug= np.column_stack([xgb_ho2, lgb_ho2, cat_ho2, ridge_ho2, knn_ho2])
test_matrix_aug   = np.column_stack([xgb_te2, lgb_te2, cat_te2, ridge_te2, knn_te2])

best_aug_loss, best_aug_w = float('inf'), None
for trial in range(100):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_matrix_aug, y_dev),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_aug_loss: best_aug_loss, best_aug_w = res.fun, res.x

ridge_meta_aug = Ridge(alpha=best_ridge_alpha); ridge_meta_aug.fit(oof_matrix_aug, y_dev)

# Aug holdout MSEs
aug_slsqp_ho = mean_squared_error(y_hold, holdout_matrix_aug @ best_aug_w)
aug_ridge_ho = mean_squared_error(y_hold, ridge_meta_aug.predict(holdout_matrix_aug))
aug_blend_ho = mean_squared_error(y_hold, 0.5*(holdout_matrix_aug @ best_aug_w) + 0.5*ridge_meta_aug.predict(holdout_matrix_aug))

In [ ]:
# AKILLI KARAR: Tüm seçeneklerden en iyi holdout MSE'yi seç
print('═'*60)
print('  PSEUDO-LABELING A/B TEST (Holdout MSE)')
print('═'*60)
all_options = [
    ('Orig + SLSQP', slsqp_ho_mse,  lambda: best_method_fn(test_matrix_p1)),
    ('Orig + Ridge', ridge_ho_mse,  lambda: ridge_meta.predict(test_matrix_p1)),
    ('Orig + Blend', blend_ho_mse,  lambda: 0.5*(test_matrix_p1 @ best_w) + 0.5*ridge_meta.predict(test_matrix_p1)),
    ('Pseudo+SLSQP', aug_slsqp_ho,  lambda: test_matrix_aug @ best_aug_w),
    ('Pseudo+Ridge', aug_ridge_ho,  lambda: ridge_meta_aug.predict(test_matrix_aug)),
    ('Pseudo+Blend', aug_blend_ho,  lambda: 0.5*(test_matrix_aug @ best_aug_w) + 0.5*ridge_meta_aug.predict(test_matrix_aug)),
]
for name, mse, _ in all_options:
    print(f'  {name:<18}: {mse:.4f}')

all_options.sort(key=lambda x: x[1])
winner_name, winner_holdout_mse, winner_test_fn = all_options[0]

# Akıllı atlama kontrolü
best_orig_mse = min(slsqp_ho_mse, ridge_ho_mse, blend_ho_mse)
best_pseudo_mse = min(aug_slsqp_ho, aug_ridge_ho, aug_blend_ho)

print('═'*60)
if best_pseudo_mse < best_orig_mse:
    improvement = best_orig_mse - best_pseudo_mse
    print(f'✓ Pseudo-labeling İYİLEŞTİRDİ ({improvement:+.4f} MSE)')
    print(f'  → Pseudo-augmented ensemble kullanılacak')
    USE_PSEUDO = True
else:
    degradation = best_pseudo_mse - best_orig_mse
    print(f'✗ Pseudo-labeling KÖTÜLEŞTİRDİ ({degradation:+.4f} MSE)')
    print(f'  → Pseudo ATLANIR, orijinal ensemble kullanılacak')
    USE_PSEUDO = False
    # Pseudo seçeneklerini filtreleme
    all_options = [o for o in all_options if 'Pseudo' not in o[0]]
    winner_name, winner_holdout_mse, winner_test_fn = all_options[0]

print(f'\n🏆 PHASE 1 KAZANAN: {winner_name}  (Holdout MSE = {winner_holdout_mse:.4f})')
print('═'*60)

final_test_phase1 = np.clip(winner_test_fn(), 0, 100)

## 12. 🚀 Phase 4 — Full Retrain on 10000 (Temizlenmiş)

In [ ]:
print('Phase 4: TÜM 10000 ile yeniden eğitim için feature engineering...\n')

# Full train + test üzerinde prepare_features (temiz, dummy hold yok)
X_full_base, X_test_base_p4, X_full_cat, X_test_cat_p4, _, _ = prepare_features(train_full, test)

# Full TE
skf_full = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_SPLITS_FULL = list(skf_full.split(train_full, y_bin_full))

te_full, te_test_p4 = {}, {}
for col in ['department','target_role','hobby','preferred_social_media_platform','university_tier']:
    oo, te = kfold_te_full(train_full, test, col, y_full, FOLD_SPLITS_FULL, 20)
    te_full[f'te_{col}'] = oo; te_test_p4[f'te_{col}'] = te

for c1, c2 in [('department','target_role'), ('university_tier','target_role'), ('target_role','hobby')]:
    combo = f'{c1}_x_{c2}'
    train_full[combo] = train_full[c1].astype(str)+'|'+train_full[c2].astype(str)
    test_t = test.copy(); test_t[combo] = test_t[c1].astype(str)+'|'+test_t[c2].astype(str)
    oo, te = kfold_te_full(train_full, test_t, combo, y_full, FOLD_SPLITS_FULL, 30)
    te_full[f'te_{combo}'] = oo; te_test_p4[f'te_{combo}'] = te
    train_full.drop(columns=[combo], inplace=True)

train_te_p4 = pd.DataFrame(te_full)
test_te_p4 = pd.DataFrame(te_test_p4)

# Birleştir
X_full = merge_all(X_full_base, nlp_full_df, train_te_p4, bert_full_df if BERT_OK else pd.DataFrame())
X_test_p4 = merge_all(X_test_base_p4, nlp_test, test_te_p4, bert_test if BERT_OK else pd.DataFrame())
X_full_cat = merge_all(X_full_cat, nlp_full_df, train_te_p4, bert_full_df if BERT_OK else pd.DataFrame())
X_test_cat_p4 = merge_all(X_test_cat_p4, nlp_test, test_te_p4, bert_test if BERT_OK else pd.DataFrame())

full_medians = X_full.median(numeric_only=True)
X_full = X_full.fillna(full_medians)
X_test_p4 = X_test_p4.fillna(full_medians)
for c in X_full_cat.columns:
    if pd.api.types.is_numeric_dtype(X_full_cat[c]) and X_full_cat[c].isna().any():
        X_full_cat[c] = X_full_cat[c].fillna(full_medians.get(c, 0))
        X_test_cat_p4[c] = X_test_cat_p4[c].fillna(full_medians.get(c, 0))

# Aynı feature subset'leri
use_cols = [c for c in all_features_keep if c in X_full.columns]
X_full_sel = X_full[use_cols]
X_test_sel_p4 = X_test_p4[use_cols]
X_full_cat_sel = X_full_cat[use_cols]
X_test_cat_sel_p4 = X_test_cat_p4[use_cols]
cat_idx_full = [X_full_cat_sel.columns.get_loc(c) for c in CAT_COLS + ['university_tier'] if c in X_full_cat_sel.columns]

use_knn = [c for c in top_30_features if c in X_full.columns]
use_poly = [c for c in top_10_features if c in X_full.columns]
scaler_full = StandardScaler()
X_full_std = pd.DataFrame(scaler_full.fit_transform(X_full), columns=X_full.columns)
X_test_std_p4 = pd.DataFrame(scaler_full.transform(X_test_p4), columns=X_test_p4.columns)
X_full_knn = X_full_std[use_knn]
X_test_knn_p4 = X_test_std_p4[use_knn]
X_full_poly = X_full[use_poly]
X_test_poly_p4 = X_test_p4[use_poly]

print(f'✓ Phase 4 features: full {X_full_sel.shape}, test {X_test_sel_p4.shape}')

In [ ]:
# Phase 4 stacking — tüm 10000 ile (holdout yok)
def stack_p4(factory, splits, X_tr, X_te, y_tr, n_seeds=N_SEEDS, kind='plain', cat_idx=None):
    oof = np.zeros(len(y_tr)); te_p = np.zeros(len(X_te))
    for seed in range(n_seeds):
        fold_te = np.zeros(len(X_te))
        for tr_idx, val_idx in splits:
            m = factory(seed)
            if kind == 'xgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])], verbose=False)
            elif kind == 'lgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            elif kind == 'cat':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=(X_tr.iloc[val_idx], y_tr[val_idx]),
                      cat_features=cat_idx, verbose=0)
            else:
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx])
            oof[val_idx] += m.predict(X_tr.iloc[val_idx]) / n_seeds
            fold_te += m.predict(X_te) / len(splits)
        te_p += fold_te / n_seeds
    return oof, te_p

print('Phase 4 Stacking (full 10000)...\n')
t0 = time.time()
f_xgb_oof, f_xgb_te = stack_p4(xgb_factory, FOLD_SPLITS_FULL, X_full_sel, X_test_sel_p4, y_full, kind='xgb')
print(f'  XGB done  ({time.time()-t0:.0f}s)')
t0 = time.time()
f_lgb_oof, f_lgb_te = stack_p4(lgb_factory, FOLD_SPLITS_FULL, X_full_sel, X_test_sel_p4, y_full, kind='lgb')
print(f'  LGB done  ({time.time()-t0:.0f}s)')
t0 = time.time()
f_cat_oof, f_cat_te = stack_p4(cat_factory, FOLD_SPLITS_FULL, X_full_cat_sel, X_test_cat_sel_p4, y_full, kind='cat', cat_idx=cat_idx_full)
print(f'  CAT done  ({time.time()-t0:.0f}s)')
t0 = time.time()
f_ridge_oof, f_ridge_te = stack_p4(ridge_factory, FOLD_SPLITS_FULL, X_full_poly, X_test_poly_p4, y_full, n_seeds=1)
print(f'  Ridge done ({time.time()-t0:.0f}s)')
t0 = time.time()
f_knn_oof, f_knn_te = stack_p4(knn_factory, FOLD_SPLITS_FULL, X_full_knn, X_test_knn_p4, y_full, n_seeds=1)
print(f'  KNN done  ({time.time()-t0:.0f}s)')

oof_full_matrix = np.column_stack([f_xgb_oof, f_lgb_oof, f_cat_oof, f_ridge_oof, f_knn_oof])
test_full_matrix = np.column_stack([f_xgb_te, f_lgb_te, f_cat_te, f_ridge_te, f_knn_te])

# Full data üzerinde weight opt
best_full_loss, best_full_w = float('inf'), None
for trial in range(100):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_full_matrix, y_full),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_full_loss: best_full_loss, best_full_w = res.fun, res.x

ridge_meta_full = Ridge(alpha=best_ridge_alpha); ridge_meta_full.fit(oof_full_matrix, y_full)

p4_slsqp_oof = (oof_full_matrix @ best_full_w)
p4_ridge_oof = ridge_meta_full.predict(oof_full_matrix)
p4_blend_oof = 0.5*p4_slsqp_oof + 0.5*p4_ridge_oof

p4_options = [
    ('Phase4 SLSQP', mean_squared_error(y_full, p4_slsqp_oof), test_full_matrix @ best_full_w),
    ('Phase4 Ridge', mean_squared_error(y_full, p4_ridge_oof), ridge_meta_full.predict(test_full_matrix)),
    ('Phase4 Blend', mean_squared_error(y_full, p4_blend_oof), 0.5*(test_full_matrix @ best_full_w) + 0.5*ridge_meta_full.predict(test_full_matrix))
]
p4_options.sort(key=lambda x: x[1])
p4_winner_name, p4_winner_mse, p4_winner_pred = p4_options[0]
final_test_phase4 = np.clip(p4_winner_pred, 0, 100)

print(f'\nPhase 4 OOF MSE (en iyi): {p4_winner_mse:.4f}  [{p4_winner_name}]')
for n, w in zip(model_names, best_full_w):
    print(f'  weight {n}: {w:.4f}')

## 13. 🏁 Final Submission (70% Phase4 + 30% Phase1)

In [ ]:
# Final = 70% Phase4 + 30% Phase1 (onaylanan strateji)
final_test = 0.7 * final_test_phase4 + 0.3 * final_test_phase1
final_test = np.clip(final_test, 0, 100)

submission = pd.DataFrame({
    'student_id'         : test_ids,
    'career_success_score': final_test
})
submission.to_csv('submission.csv', index=False)

# Alternatif submissions
pd.DataFrame({'student_id': test_ids, 'career_success_score': final_test_phase1}).to_csv('submission_phase1.csv', index=False)
pd.DataFrame({'student_id': test_ids, 'career_success_score': final_test_phase4}).to_csv('submission_phase4.csv', index=False)

print('═'*60)
print('  🏆 v6 FINAL RESULTS')
print('═'*60)
print(f'  Phase 1 Holdout MSE (validated)    : {winner_holdout_mse:.4f}')
print(f'  Phase 4 OOF MSE (full retrain)     : {p4_winner_mse:.4f}')
print(f'  Pseudo-labeling kullanıldı mı?     : {"EVET" if USE_PSEUDO else "HAYIR"}')
print(f'  Final blend ratio                  : 70% Phase4 + 30% Phase1')
print('-'*60)
print(f'  Leaderboard referans:')
print(f'    1. sıra (strawveri)              : 80.99')
print(f'    10. sıra (ÖzRüm)                 : 82.46')
print(f'    Sen (v1)                         : 89.54')
best_est = min(winner_holdout_mse, p4_winner_mse)
print(f'    v6 beklenen (tahmini)            : ~{best_est:.2f}')
print('═'*60)
print(f'\n3 submission dosyası üretildi:')
print(f'  ⭐ submission.csv         (70/30 blend - ÖNCE BUNU SUBMIT ET)')
print(f'     submission_phase1.csv  (sadece dev, holdout-validated)')
print(f'     submission_phase4.csv  (full 10000 retrain)')
print(f'\n{submission.head()}')

In [ ]:
# Görselleştirme
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Per-model dev OOF MSE vs Holdout MSE
dev_mses = [mean_squared_error(y_dev, oof_matrix[:,i]) for i in range(n_models)]
hold_mses = [mean_squared_error(y_hold, holdout_matrix[:,i]) for i in range(n_models)]
x = np.arange(len(model_names))
axes[0,0].bar(x - 0.2, dev_mses, 0.4, label='Dev OOF MSE', color='steelblue')
axes[0,0].bar(x + 0.2, hold_mses, 0.4, label='Holdout MSE', color='coral')
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(model_names)
axes[0,0].legend(); axes[0,0].set_title('OOF vs Holdout — Per Model')
axes[0,0].set_ylabel('MSE')

# Dağılım
axes[0,1].hist(y_full, bins=50, alpha=0.6, color='steelblue', label='Train', edgecolor='white')
axes[0,1].hist(final_test, bins=50, alpha=0.6, color='coral', label='Test (final)', edgecolor='white')
axes[0,1].set_title('Tahmin Dağılımı vs Train')
axes[0,1].legend()

# Stability
fold_data = []
for name, fmse in stability.items():
    for v in fmse: fold_data.append({'Model': name, 'MSE': v})
fold_df = pd.DataFrame(fold_data)
sns.boxplot(data=fold_df, x='Model', y='MSE', ax=axes[1,0])
axes[1,0].set_title('Per-Fold MSE Stability')

# Phase4 Ensemble Weights
axes[1,1].pie(best_full_w, labels=model_names, autopct='%1.1f%%')
axes[1,1].set_title('Phase 4 Ensemble Weights')

plt.tight_layout(); plt.show()

---
## 📋 v6 Mimari Özet

| Bölüm | Detay |
|-------|--------|
| **Validation** | 80/20 stratified hold-out + 10-fold StratifiedKFold |
| **NLP** | TF-IDF (word + char) + Sentence-BERT (multilingual) |
| **Target Encoding** | Single + Cross-categorical, K-fold smoothed |
| **Feature Selection** | XGB importance — top %85 tree, top 30 KNN, top 10 Ridge |
| **Base Models (5)** | XGB, LGB, CAT (trees) + Ridge+Poly (linear) + KNN (distance) |
| **Optuna Trials** | XGB:60, LGB:60, CAT:40, Ridge:15, KNN:15 |
| **Stacking** | 10-fold × 3 seed, multi-seed averaging |
| **Meta-learning** | SLSQP + Ridge + Blend — holdout MSE'ye göre seç |
| **Pseudo-Labeling** | Akıllı: holdout iyileşmezse atlanır |
| **Final** | 70% Phase4 (full retrain) + 30% Phase1 (validated) |

## ⏱️ Beklenen Süre (GPU)
- Setup + BERT + features: ~10 dk
- Optuna (190 trial): ~25 dk
- Phase 1 stacking: ~10 dk
- Pseudo stacking: ~10 dk
- Phase 4 stacking: ~10 dk
- **Toplam: ~30-45 dk**

## 🎯 Beklenen Sonuç
- v1 (mevcut): 89.54 MSE (117. sıra)
- **v6 hedef**: ~83-85 MSE (top 10-20 arası)

## 🚀 Submission Stratejisi
1. **İlk olarak `submission.csv` submit et** (70/30 blend, en güvenli)
2. Eğer public LB iyi gelirse → tamam
3. Eğer hayal kırıklığıysa → `submission_phase4.csv` dene
4. Eğer o da iyi değilse → `submission_phase1.csv` dene